<!-- beginner-banner-v1 -->

> 🧭 **비개발자 수강생 안내** — 이 노트북에서 새로 배우는 것: 한 질문을 **풍부한 검색어로 다시 쓰기** — HyDE / Multi-Query / 분해.
>
> - 📖 강의 페이지: [day3/17-advanced-rag-query](https://siapapa.github.io/day3/17-advanced-rag-query/)
> - 🆕 처음이라면 → [비개발자 학습 가이드](https://siapapa.github.io/beginners-guide/)
> - 🔤 모르는 단어 → [용어 사전](https://siapapa.github.io/appendix/glossary/)
> - 🛠️ 환경/접속 막힘 → [사전 준비](https://siapapa.github.io/setup/) · [트러블슈팅](https://siapapa.github.io/appendix/troubleshooting/)
>
> **셀은 위에서 아래로 차례대로 실행**하세요. 시연용 코드(`구경만 하세요` 표시)는 지금 이해 못 해도 100% 정상입니다.

---



# 14. Advanced RAG — 쿼리 변환 (HyDE · Multi-Query · Query Decomposition)
> Day 3 · 17H · 소요 약 50분

## 학습 목표

- **Naive RAG** 의 한계 — 질문과 문서의 어휘가 다르면 검색이 빗나가는 상황을 재현한다.
- **HyDE** (Hypothetical Document Embeddings) 로 가상 답변을 만들어 검색 정확도를 높인다.
- **Multi-Query Retriever** 로 같은 질문을 여러 관점으로 자동 변형하여 재현율을 올린다.
- **Query Decomposition** 으로 복잡한 질문을 하위 질문 2~4 개로 분해해 통합 검색한다.
- 동일한 질문 셋에 대해 세 기법의 결과를 **한눈에 비교**한다.

> **DB 미사용.** 이 노트북은 Neon 에 붙지 않고 13 번과 유사한 병원 안내 문서 코퍼스를 다시 만들어 Chroma 에 적재합니다. 퍼시스트 경로는 `./rag_chroma` — 04/05/10/13 의 경로와 분리합니다.

In [ ]:
%pip install -q langchain langchain-openai langchain-community langchain-core langchain-chroma chromadb

In [ ]:
# Colab/로컬 환경에서 필요한 환경변수를 안전하게 로딩합니다 (다른 노트북과 동일 패턴).
import os

def _load_secret(key: str, required: bool = True) -> None:
    """Colab Secrets → getpass 입력 순으로 시도해 환경변수에 적재."""
    if os.environ.get(key):
        return
    value = None
    try:
        from google.colab import userdata  # type: ignore
        value = userdata.get(key)
    except Exception:
        value = None
    if not value:
        try:
            from getpass import getpass
            value = getpass(f"Enter {key}: ")
        except Exception:
            value = None
    if value:
        os.environ[key] = value
    elif required:
        raise RuntimeError(f"{key} is not set. Register it in Colab Secrets or via env var.")

# 임베딩 + LLM 모두 OpenAI 키 하나로 처리.
_load_secret("OPENAI_API_KEY", required=True)
print("Environment ready.")

## 1. 문서 코퍼스 + Chroma 벡터스토어 재구축

질문 변환 기법 3 종을 비교하려면 **동일한 검색 인프라**가 필요합니다. 13 번과 유사한 병원 안내 문서를 약간 확장하여 `./rag_chroma` 경로에 적재합니다. `collection_name` 도 `query_rewrite` 로 분리하여 다른 노트북과 충돌하지 않습니다.

In [ ]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.documents import Document

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

hospital_documents = [
    Document(page_content="내과는 평일 09:00-18:00 외래 진료를 하며 김철수(심장내과), 이영희(호흡기내과), 신민아(소화기내과) 전문의가 환자 수 기준 원내 1~3위를 차지합니다.", metadata={"dept": "내과", "type": "doctor"}),
    Document(page_content="외과에는 박민수(일반외과), 정수진(흉부외과), 권혁준(혈관외과) 전문의가 근무하며 일반외과가 가장 수술 건수가 많습니다.", metadata={"dept": "외과", "type": "doctor"}),
    Document(page_content="소아과에는 최동현, 강미래, 문서영 전문의가 있으며 영유아 예방접종부터 청소년 성장 클리닉까지 운영합니다.", metadata={"dept": "소아과", "type": "doctor"}),
    Document(page_content="정형외과에는 윤성호(척추외과), 한지은(관절외과) 전문의가 근무하며 디스크·오십견 환자가 많습니다.", metadata={"dept": "정형외과", "type": "doctor"}),
    Document(page_content="진료 시간은 평일 09:00-18:00, 토요일 09:00-13:00 이며 점심시간 12:30-13:30 에는 외래 접수가 중단됩니다.", metadata={"type": "schedule"}),
    Document(page_content="응급실은 24시간 운영됩니다. 야간 당직은 내과·외과 각 1명이 상주하며 심장내과 김철수 전문의는 매주 수요일 야간 당직을 섭니다.", metadata={"type": "emergency"}),
    Document(page_content="입원 병실 요금은 1인실 250,000원/일, 2인실 150,000원/일, 4인실 80,000원/일이며 식대는 별도 1식 8,000원입니다.", metadata={"type": "admission"}),
    Document(page_content="외래 환자는 주차 3시간 무료, 이후 30분당 1,000원입니다. 입원 보호자는 1일 5,000원 정액제입니다.", metadata={"type": "parking"}),
    Document(page_content="심장 질환 응급환자는 야간에도 심장내과 당직의가 1차 평가 후 필요 시 심혈관 중재술 팀을 호출합니다.", metadata={"type": "emergency", "dept": "심장내과"}),
    Document(page_content="김철수 전문의는 심장내과 15년차 경력으로 관상동맥 중재술 1,200례를 보유하고 있으며 원내 최다 경력자입니다.", metadata={"dept": "심장내과", "type": "doctor_detail"}),
]

vectorstore = Chroma.from_documents(
    hospital_documents,
    embeddings,
    collection_name="query_rewrite",
    persist_directory="./rag_chroma",
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print(f"Vectorstore ready: {len(hospital_documents)} docs, top-k=3")

## 2. Naive RAG — 왜 질문 변환이 필요한가

질문과 문서가 같은 단어를 쓰지 않으면 벡터 검색도 놓칠 수 있습니다. 먼저 **원본 질문 그대로** 검색해 보고 결과가 어떻게 나오는지 확인합니다.

In [ ]:
naive_question = "재방문율이 높은 진료과는?"  # 문서에는 "재방문율"이라는 단어가 없음

naive_results = retriever.invoke(naive_question)
print(f"❓ {naive_question}")
print(f"🔵 Naive 검색 결과 ({len(naive_results)}개):")
for i, doc in enumerate(naive_results):
    print(f"  [{i+1}] {doc.page_content[:80]}...")

## 3. HyDE — Hypothetical Document Embeddings

**핵심 직관:** "질문보다 답변이 문서와 더 비슷하다."

질문 → LLM 으로 **가상의 답변 문서** 생성 → 가상 답변을 임베딩해서 검색.
문서 코퍼스와 어휘가 비슷한 "답변 형태" 로 검색하므로 용어 불일치를 해소합니다.

In [ ]:
# HyDE = Hypothetical Document Embeddings (가상 문서 임베딩).
# 직관: "질문" 보다 "그 질문의 답변" 이 실제 문서와 어휘적으로 더 비슷합니다.
# 그래서 LLM 으로 "가짜 답변" 을 한 번 만들고, 그것을 임베딩해서 검색에 사용합니다.
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 일부러 "실제 데이터가 없어도 괜찮다" 라고 명시 — LLM 이 "모릅니다" 라고 거부하지 않게.
hyde_prompt = ChatPromptTemplate.from_template(
    "다음 질문에 대한 답변이 포함된 문서를 작성하세요.\n"
    "실제 데이터가 없어도 괜찮습니다. 문서 형태로 가상의 답변을 만들어주세요.\n\n"
    "질문: {question}\n\n"
    "가상 문서:"
)

# LCEL 파이프 — 프롬프트 → LLM → 문자열 변환.
hyde_chain = hyde_prompt | llm | StrOutputParser()

question = "병원에서 가장 바쁜 진료과는 어디인가요?"
hypothetical_doc = hyde_chain.invoke({"question": question})
print(f"❓ 질문: {question}\n")
print(f"📄 가상 문서:\n{hypothetical_doc}\n")

In [ ]:
# 가상 문서로 검색 vs 원 질문으로 검색
hyde_results = vectorstore.similarity_search(hypothetical_doc, k=3)
print("🟢 HyDE (가상 문서로 검색) 결과:")
for i, doc in enumerate(hyde_results):
    print(f"  [{i+1}] {doc.page_content[:80]}...")

naive_results = vectorstore.similarity_search(question, k=3)
print("\n🔵 Naive (원 질문으로 검색) 결과:")
for i, doc in enumerate(naive_results):
    print(f"  [{i+1}] {doc.page_content[:80]}...")

## 4. Multi-Query Retriever — 질문 다각화

LLM 이 원 질문을 **3~5 개의 변형 질문** 으로 자동 재작성하고, 각 변형으로 검색한 결과를 **합집합** (중복 제거) 으로 돌려줍니다.

`logging` 을 `INFO` 레벨로 올리면 LLM 이 실제로 만든 변형 질문들을 눈으로 확인할 수 있습니다.

In [ ]:
# Multi-Query Retriever — LLM 이 원 질문을 3~5 개 변형으로 자동 재작성.
# 각 변형 질문으로 검색한 결과를 모아 중복 제거 후 합집합으로 돌려줍니다.
import logging
from langchain.retrievers.multi_query import MultiQueryRetriever

# logging 레벨을 INFO 로 올리면 LLM 이 실제로 만든 변형 질문이 콘솔에 찍혀 학생이 눈으로 확인 가능.
logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

# .from_llm(retriever, llm) 한 줄로 변형 질문 생성용 LLM 까지 같이 묶어 줍니다.
multi_retriever = MultiQueryRetriever.from_llm(
    retriever=retriever,   # 기존 vectorstore retriever 를 그대로 활용
    llm=llm,               # 변형 질문 생성에 사용
)

mq_question = "입원하면 비용이 얼마나 드나요?"
multi_results = multi_retriever.invoke(mq_question)

print(f"\n❓ 원래 질문: {mq_question}")
print(f"🟡 Multi-Query 검색 결과 ({len(multi_results)}개, 중복 제거 후):")
for i, doc in enumerate(multi_results):
    print(f"  [{i+1}] {doc.page_content[:80]}...")

## 5. Query Decomposition — 복잡한 질문 분해

"A 는 어떻고, B 는 어떠며, 둘의 차이는?" 처럼 한 문장에 **여러 하위 질문**이 섞인 경우, 통째로 검색하면 어느 것도 제대로 맞추지 못합니다.
LLM 에게 **2~4 개의 단순 하위 질문**으로 분해시키고, 각 하위 질문으로 따로 검색해서 합칩니다.

In [ ]:
decompose_prompt = ChatPromptTemplate.from_template(
    "다음 복잡한 질문을 2-4개의 단순한 하위 질문으로 분해하세요.\n"
    "각 하위 질문은 한 줄에 하나씩 작성하세요. 번호나 기호는 붙이지 않아도 됩니다.\n\n"
    "복잡한 질문: {question}\n\n"
    "하위 질문:"
)

decompose_chain = decompose_prompt | llm | StrOutputParser()

complex_question = "내과와 외과 중 어느 쪽이 더 많은 의사가 있고, 각 과의 전문 분야는 무엇인가요?"
sub_questions_text = decompose_chain.invoke({"question": complex_question})
print(f"❓ 복잡한 질문: {complex_question}")
print(f"\n📋 하위 질문 (raw):\n{sub_questions_text}")

In [ ]:
# 자유 텍스트 응답을 줄 단위로 파싱 — "1) ...", "2. ..." 같은 머리말을 깔끔히 제거.
sub_q_list = [
    # lstrip 의 인자는 "이 문자들 중 어느 것이든 처음부터 제거" 의미.
    # 0123456789 → 숫자, .-)• → 기호, 마지막 공백까지.
    q.strip().lstrip("0123456789.-)• ").strip()
    for q in sub_questions_text.split("\n")
    if q.strip()   # 빈 줄 제거
]
print(f"\n파싱된 하위 질문 {len(sub_q_list)}개:")
# enumerate(..., 1) 는 1부터 카운트 시작 — 사람이 읽기 좋게.
for i, sq in enumerate(sub_q_list, 1):
    print(f"  {i}. {sq}")

# 각 하위 질문으로 따로 검색한 뒤 결과를 한 리스트로 누적.
all_results = []
for sq in sub_q_list:
    results = retriever.invoke(sq)
    all_results.extend(results)
    print(f"\n🔍 '{sq[:50]}...' → {len(results)}개")

# 중복 제거 — page_content 문자열을 set 으로 추적하며 처음 보는 것만 보관.
# (Document 객체끼리는 == 비교가 동일 인스턴스만 같다고 처리되므로 본문 문자열로 비교)
unique_contents = set()
unique_results = []
for doc in all_results:
    if doc.page_content not in unique_contents:
        unique_contents.add(doc.page_content)
        unique_results.append(doc)

print(f"\n📊 통합 결과: {len(unique_results)}개 (중복 제거 후)")
for i, doc in enumerate(unique_results, 1):
    print(f"  [{i}] {doc.page_content[:70]}...")

### 구조화 출력 대안 (프로덕션 팁)

위 예시는 "한 줄에 하나씩" 자유 텍스트를 받아 `split("\n")` + `lstrip(...)` 으로 파싱합니다. 수업 시간엔 **포맷 깨짐을 체감** 시키려고 일부러 자유 텍스트 방식을 먼저 보여줍니다.

본인 프로젝트에 적용할 때는 **12H 에서 배운** `with_structured_output` 으로 `List[str]` 을 직접 받으세요 — 번호/빈 줄/머리말 파싱이 필요 없어집니다.

```python
from pydantic import BaseModel, Field

class SubQuestions(BaseModel):
    questions: list[str] = Field(description="2-4개의 단순 하위 질문")

structured = llm.with_structured_output(SubQuestions)
sub_q_list = structured.invoke(decompose_prompt.format(question=complex_question)).questions
```

## 6. 세 기법 비교 실험

동일 질문 5 개에 대해 **Naive / HyDE / Multi-Query** 를 한 번에 돌리고 결과 수·Top-1 문서를 표로 정리합니다.

> Decomposition 은 "복합 질문" 에만 어울리므로 별도 칸에서 확인합니다 (아래 셀 2 개).

In [ ]:
import pandas as pd

compare_questions = [
    "야간에 응급 진료를 받으려면 어떻게 하나요?",
    "가장 경험이 많은 의사는 누구인가요?",
    "입원 1인실은 얼마인가요?",
    "주차 요금은 어떻게 되나요?",
    "재방문율이 높은 진료과는?",  # 문서 어휘와 불일치
]

rows = []
for q in compare_questions:
    naive = retriever.invoke(q)
    hypo = hyde_chain.invoke({"question": q})
    hyde = vectorstore.similarity_search(hypo, k=3)
    multi = multi_retriever.invoke(q)

    rows.append({
        "question": q,
        "naive_n": len(naive),
        "naive_top1": naive[0].page_content[:40] if naive else "",
        "hyde_n": len(hyde),
        "hyde_top1": hyde[0].page_content[:40] if hyde else "",
        "mq_n": len(multi),
        "mq_top1": multi[0].page_content[:40] if multi else "",
    })

df = pd.DataFrame(rows)
df

In [ ]:
# Decomposition 전용 — 복합 질문 2개
decomp_questions = [
    "내과와 외과 중 어느 쪽이 더 많은 의사가 있고, 각 과의 전문 분야는 무엇인가요?",
    "야간 당직 제도가 어떻게 운영되고 심장 응급환자는 어떤 절차를 밟나요?",
]

for cq in decomp_questions:
    print(f"\n{'='*60}\n❓ {cq}\n{'='*60}")

    # Naive (비교)
    naive = retriever.invoke(cq)
    print(f"🔵 Naive ({len(naive)}개):")
    for doc in naive:
        print(f"  - {doc.page_content[:60]}...")

    # Decomposition
    raw = decompose_chain.invoke({"question": cq})
    subs = [q.strip().lstrip("0123456789.-)• ").strip() for q in raw.split("\n") if q.strip()]
    print(f"\n🟣 분해된 하위 질문 {len(subs)}개:")
    for s in subs:
        print(f"  ▸ {s}")

    # 통합 검색
    seen = set()
    merged = []
    for sq in subs:
        for doc in retriever.invoke(sq):
            if doc.page_content not in seen:
                seen.add(doc.page_content)
                merged.append(doc)
    print(f"\n🟣 Decomposition 통합 결과 ({len(merged)}개):")
    for doc in merged:
        print(f"  - {doc.page_content[:60]}...")

## 7. 해석 가이드 — 어떤 기법이 어떤 질문에 강한가?

| 기법 | 강점 | 약점 | 추가 비용 |
|---|---|---|---|
| **Naive** | 단순·빠름 | 어휘 불일치에 매우 취약 | 없음 (검색만) |
| **HyDE** | 추상적·용어 불일치 질문에 강함 | 가상 문서가 엉뚱하면 오히려 노이즈 | LLM 1 회 호출 |
| **Multi-Query** | 재현율 향상, 가장 범용적 | 변형이 유사하면 중복만 늘어남 | LLM 1 회 + 검색 3~5 회 |
| **Decomposition** | **복합 질문**에 강함 | 단순 질문에 쓰면 과분해 | LLM 1 회 + 검색 2~4 회 |

**프로덕션 조합 패턴:**

- Multi-Query + Re-rank (다음 15 번 노트북) — 재현율 ↑ + 정밀도 ↑
- HyDE + Hybrid 검색 (15 번) — 의미·키워드 양쪽 보완
- Decomposition + Multi-Query — 복합 질문을 쪼개고 각 조각을 다각화

## 실습 과제

다음 1 가지 실습을 직접 작성해 보세요. (정답 코드는 의도적으로 비워 두었습니다.)

### 1. 본인 프로젝트 질문으로 3기법 비교
본인 프로젝트의 도메인에 맞는 질문 2개를 선택하여 3기법을 비교하세요.

위에서 정의한 `compare_retrieval(question)` 함수를 본인 도메인 질문 2개에 대해 호출하고, Naive / HyDE / Multi-Query 결과 수를 비교하세요.

_힌트: 질문은 어휘 불일치(추상적 질문)와 구체적 질문(고유명사 포함) 한 개씩 섞으면 기법별 차이를 더 잘 관찰할 수 있습니다._

**비교 결과를 아래 표에 기록하세요:**

| 질문 | Naive 결과 수 | HyDE 결과 수 | Multi-Query 결과 수 | 가장 좋은 기법 |
|---|---|---|---|---|
| (질문 1) | | | | |
| (질문 2) | | | | |

어떤 기법이 본인 도메인에서 가장 좋은 결과를 보이나요?


In [ ]:
# ============================================================
# 실습 과제 — 본인 프로젝트 질문으로 3기법 비교
# ============================================================

# 실습 1: 본인 도메인 질문 2개로 Naive / HyDE / Multi-Query 비교
# TODO: compare_retrieval(question) 함수를 본인 도메인 질문 2개에 대해 호출하고 결과 수를 표로 정리하세요.
# 여기에 구현하세요.


## 다음 노트북에서는…

**`15_advanced_rag_retrieval.ipynb`** 에서 **검색 자체의 품질**을 올립니다 — **BM25 + 벡터 하이브리드** (키워드 정밀 + 의미 재현율), **CrossEncoder Re-rank** (2 단계: 후보 생성 → 재정렬), 그리고 **Parent-Child Chunking** (작은 청크로 검색, 큰 청크로 컨텍스트 제공) 의 조합 효과를 측정합니다.